# 02 — Train classical baselines: smoke (tiny RF + MLflow)

**SYNTHETIC / SMOKE ONLY — NOT real labels.** Trains `geoeco.train.train_classical`
spatial CV on the 300-row synthetic table from notebook 01 (seed 42) with a tiny RF
(`n_estimators=10`, 1×1 param grid) so a free Kaggle CPU finishes in ~1–2 min.
Logs params/metrics to MLflow and prints the metrics dict.

Real run (needs QGIS labels `qgis-hyd v2` + point table) is printed — not executed — in the
last cell. M6 UNet needs a GPU notebook (see `kaggle/KAGGLE_RUNBOOK.md` §4).

In [ ]:
# Cell 1 — Kaggle/CPU setup (idempotent) + MLflow file-store opt-in for mlflow>=3.
import os
import subprocess
import sys
import time
from pathlib import Path

os.environ.setdefault("MLFLOW_ALLOW_FILE_STORE", "true")  # mlflow 3.x file backend gate
os.environ.setdefault("MLFLOW_DISABLE_AGENT_HINT", "1")


def find_repo() -> Path:
    cands = [Path.cwd(), Path.cwd() / "PrithviBodh",
             Path("/kaggle/working/PrithviBodh"), Path("/kaggle/working")]
    for c in cands:
        if (c / "configs" / "aoi" / "hyderabad.yaml").exists():
            return c
    subprocess.run(["git", "clone", "https://github.com/veeresh0804/PrithviBodh.git"],
                   check=True, cwd=str(Path.cwd()))
    return Path.cwd() / "PrithviBodh"


REPO = find_repo()
if str(REPO) not in sys.path:
    sys.path.insert(0, str(REPO))
MLFLOW_DIR = Path("/kaggle/working/mlruns") if Path("/kaggle").exists() else REPO / "mlruns"
print("REPO =", REPO, "| MLFLOW_DIR =", MLFLOW_DIR)

import importlib
for _mod, _pip in [("sklearn", "scikit-learn"), ("mlflow", "mlflow"),
                   ("yaml", "pyyaml"), ("pandas", "pandas"),
                   ("numpy", "numpy"), ("pyarrow", "pyarrow")]:
    try:
        importlib.import_module(_mod)
    except ImportError:
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", _pip], check=True)
print("setup ok")

In [ ]:
# Cell 2 — Load configs (AOI Hyderabad + Sentinel spec + RF real-vs-smoke params).
from geoeco.utils.config import load_yaml_config, require_keys

aoi = load_yaml_config(REPO / "configs" / "aoi" / "hyderabad.yaml")
require_keys(aoi, ["bounds_wgs84"], name="hyderabad.yaml")
sent = load_yaml_config(REPO / "configs" / "data" / "sentinel.yaml")
assert sent["stack_per_season"] == 20, "sentinel contract drift"
lab = load_yaml_config(REPO / "configs" / "labels" / "labelling.yaml")
cv = load_yaml_config(REPO / "configs" / "eval" / "spatial_cv.yaml")
assert lab["seed"] == cv.get("seed") == 42, "seed gate failed"
rf = load_yaml_config(REPO / "configs" / "model" / "rf.yaml")
print("AOI:", aoi["bounds_wgs84"], "| seed:", lab["seed"])
print(f"REAL RF (configs/model/rf.yaml): n_estimators={rf['rf']['n_estimators']}, "
      f"grid={list(__import__('geoeco.models.classical', fromlist=['RF_PARAM_GRID']).RF_PARAM_GRID)}")
print("SMOKE override this notebook: n_estimators=10, grid=max_features=[sqrt] x min_samples_leaf=[1]")

In [ ]:
# Cell 3 — Rebuild the SYNTHETIC/SMOKE point table inline (self-contained, seed 42).
# Same recipe as nb 01 cell 4: real skeleton geometry + random features/labels.
# 60 pts x 5 blocks (B0-B4) = 300 rows so GroupKFold(n_splits=5) is satisfied.
import numpy as np
import pandas as pd

from geoeco.labels.pipeline import build_skeleton, select_overlap
from geoeco.models.classical import feature_columns

SMOKE_SEED = 42
pts, counts = build_skeleton()
select_overlap(pts, float(lab["overlap_fraction"]), int(lab["seed"]))
cols = feature_columns("M3")
assert len(cols) == 60, f"M3 contract drift: {len(cols)} cols"

rng = np.random.default_rng(SMOKE_SEED)
sub = []
for b in sorted({p["block"] for p in pts}):
    sub += [p for p in pts if p["block"] == b][:60]
n = len(sub)
X = rng.normal(size=(n, len(cols))).astype(np.float32)
y = rng.integers(0, 6, size=n)  # SYNTHETIC labels 0..5 (NOT human, NOT DW/WorldCover)
X += y[:, None].astype(np.float32) * 0.3
df = pd.DataFrame(X, columns=cols)
df["label"] = y
df["block_id"] = [p["block"] for p in sub]
df["synthetic"] = True
assert len(set(df["block_id"].tolist())) >= 5
print("smoke table:", df.shape, "| per-block:", df["block_id"].value_counts().to_dict())

In [ ]:
# Cell 4 — Run train_classical spatial CV (tiny RF, CPU) + MLflow, print metrics dict.
# Real code path: geoeco.train.train_classical.run_spatial_cv (5-fold GroupKFold over
# blocks, inner spatial GridSearchCV, MLflow params/metrics). Smoke patch: 10 trees +
# 1x1 grid. model-artifact logging is best-effort (mlflow 3.x skops gate) with a
# joblib fallback; params/metrics ALWAYS land in MLflow.
import mlflow
import mlflow.sklearn

import geoeco.train.train_classical as tc

_orig_build_rf = tc.build_rf
tc.build_rf = lambda random_state=42, **kw: _orig_build_rf(n_estimators=10,
                                                           random_state=random_state)
tc.RF_PARAM_GRID = {"max_features": ["sqrt"], "min_samples_leaf": [1]}

_orig_log_model = mlflow.sklearn.log_model


def _safe_log_model(*a, **k):
    try:
        return _orig_log_model(*a, **k)
    except Exception as e:  # mlflow 3.x trusted-types gate; metrics already logged
        print("NOTE: mlflow.sklearn.log_model skipped:", type(e).__name__, "-> joblib fallback")
        import joblib
        import tempfile
        est = a[0] if a else k.get("sk_model")
        with tempfile.TemporaryDirectory() as td:
            p = os.path.join(td, "rf_smoke.joblib")
            joblib.dump(est, p)
            mlflow.log_artifact(p, artifact_path="rf-fallback")
        return None


mlflow.sklearn.log_model = _safe_log_model
mlflow.set_tracking_uri("file:" + MLFLOW_DIR.as_posix())

t0 = time.time()
summary = tc.run_spatial_cv(df, "M3", experiment="geoeco-classical-smoke", seed=SMOKE_SEED)
wall = time.time() - t0
metrics = {k: v for k, v in summary.items()}
print(f"wall: {wall:.1f}s (budget < 300s)")
print("metrics dict:", metrics)

In [ ]:
# Cell 5 — MLflow readback + REAL-LABELS next steps (printed, NOT executed here).
try:
    from mlflow.tracking import MlflowClient
    client = MlflowClient(tracking_uri="file:" + MLFLOW_DIR.as_posix())
    exp = client.get_experiment_by_name("geoeco-classical-smoke")
    runs = client.search_runs(exp.experiment_id, order_by=["metrics.macro_f1_mean DESC"],
                              max_results=1)
    print("logged:", runs[0].data.metrics)
except Exception as e:
    print("readback skipped:", type(e).__name__, "— check", MLFLOW_DIR)

assert "mean" in summary and "fold_macro_f1" in summary and len(summary["fold_macro_f1"]) == 5
print(f"SMOKE macro-F1 mean={summary['mean']:.4f} +- {summary['std']:.4f} (synthetic, expect ~0.3-0.5)")
print()
print("REAL-LABELS next steps (needs qgis-hyd v2 + point table + GPU for M6):")
print("  1. for M in M1 M2 M3 M3-lgbm M5; do python -m geoeco.train.train_classical"
      " --points data/processed/points.parquet --model $M --seed 42; done")
print("  2. python -m geoeco.train.train_seg --stage hyd_finetune --data data/processed/chips/"
      " --model M6 --epochs 100 --batch 8   # unet.yaml: epochs [50,100], batch [8,16]")
print("  3. mlruns/ -> /kaggle/working/mlruns -> Save Output. See kaggle/KAGGLE_RUNBOOK.md §4.")
print("02 DONE.")